IMPORTS 

In [1]:
import pandas as pd
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

BASE_DIR = Path("..")

RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "Online Retail.xlsx"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
TABLES_DIR = BASE_DIR / "outputs" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

Setup complete.


lOAD RAW DATASET

In [2]:
df = pd.read_excel(RAW_DATA_PATH)

print("Raw dataset shape:", df.shape)

df.head()

Raw dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


ORIGINAL DATA COUNT

In [3]:
quality_before = {
    "Total rows": len(df),
    "Missing CustomerID": df["CustomerID"].isnull().sum(),
    "Missing Description": df["Description"].isnull().sum(),
    "Cancelled invoices": df["InvoiceNo"].astype(str).str.startswith("C").sum(),
    "Negative Quantity": (df["Quantity"] < 0).sum(),
    "Zero or Negative UnitPrice": (df["UnitPrice"] <= 0).sum(),
    "Duplicate Rows": df.duplicated().sum()
}

quality_before_df = pd.DataFrame(
    quality_before.items(),
    columns=["Issue", "Before_Cleaning"]
)

quality_before_df

,Issue,Before_Cleaning
0,Total rows,541909
1,Missing CustomerID,135080
2,Missing Description,1454
3,Cancelled invoices,9288
4,Negative Quantity,10624
5,Zero or Negative UnitPrice,2517
6,Duplicate Rows,5268


CLEAN DATASET

In [4]:
cleaned_df = df.copy()

# Remove missing CustomerID because customer segmentation requires customer identity
cleaned_df = cleaned_df.dropna(subset=["CustomerID"])

# Remove cancelled invoices
cleaned_df = cleaned_df[
    ~cleaned_df["InvoiceNo"].astype(str).str.startswith("C")
]

# Remove invalid quantity and price records
cleaned_df = cleaned_df[cleaned_df["Quantity"] > 0]
cleaned_df = cleaned_df[cleaned_df["UnitPrice"] > 0]

# Remove duplicate rows
cleaned_df = cleaned_df.drop_duplicates()

# Convert data types
cleaned_df["InvoiceDate"] = pd.to_datetime(cleaned_df["InvoiceDate"])
cleaned_df["CustomerID"] = cleaned_df["CustomerID"].astype(int)

# Create total transaction value
cleaned_df["TotalPrice"] = cleaned_df["Quantity"] * cleaned_df["UnitPrice"]

print("Cleaned dataset shape:", cleaned_df.shape)

cleaned_df.head()

Cleaned dataset shape: (392692, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


DATA QUALITY AFTER CLEANING

In [5]:
quality_after = {
    "Total rows": len(cleaned_df),
    "Missing CustomerID": cleaned_df["CustomerID"].isnull().sum(),
    "Missing Description": cleaned_df["Description"].isnull().sum(),
    "Cancelled invoices": cleaned_df["InvoiceNo"].astype(str).str.startswith("C").sum(),
    "Negative Quantity": (cleaned_df["Quantity"] < 0).sum(),
    "Zero or Negative UnitPrice": (cleaned_df["UnitPrice"] <= 0).sum(),
    "Duplicate Rows": cleaned_df.duplicated().sum()
}

quality_after_df = pd.DataFrame(
    quality_after.items(),
    columns=["Issue", "After_Cleaning"]
)

quality_after_df

,Issue,After_Cleaning
0,Total rows,392692
1,Missing CustomerID,0
2,Missing Description,0
3,Cancelled invoices,0
4,Negative Quantity,0
5,Zero or Negative UnitPrice,0
6,Duplicate Rows,0


COMPARE 

In [6]:
cleaning_summary = quality_before_df.merge(
    quality_after_df,
    on="Issue",
    how="left"
)

cleaning_summary["Removed_or_Resolved"] = (
    cleaning_summary["Before_Cleaning"] - cleaning_summary["After_Cleaning"]
)

cleaning_summary.to_csv(
    TABLES_DIR / "cleaning_summary.csv",
    index=False
)

cleaning_summary

,Issue,Before_Cleaning,After_Cleaning,Removed_or_Resolved
0,Total rows,541909,392692,149217
1,Missing CustomerID,135080,0,135080
2,Missing Description,1454,0,1454
3,Cancelled invoices,9288,0,9288
4,Negative Quantity,10624,0,10624
5,Zero or Negative UnitPrice,2517,0,2517
6,Duplicate Rows,5268,0,5268


CLEANED DATASET OVERVIEW

In [7]:
cleaned_overview = pd.DataFrame({
    "Metric": [
        "Cleaned rows",
        "Cleaned columns",
        "Unique customers",
        "Unique invoices",
        "Unique products",
        "Unique countries",
        "Start date",
        "End date",
        "Total revenue"
    ],
    "Value": [
        cleaned_df.shape[0],
        cleaned_df.shape[1],
        cleaned_df["CustomerID"].nunique(),
        cleaned_df["InvoiceNo"].nunique(),
        cleaned_df["StockCode"].nunique(),
        cleaned_df["Country"].nunique(),
        cleaned_df["InvoiceDate"].min(),
        cleaned_df["InvoiceDate"].max(),
        round(cleaned_df["TotalPrice"].sum(), 2)
    ]
})

cleaned_overview.to_csv(
    TABLES_DIR / "cleaned_dataset_overview.csv",
    index=False
)

cleaned_overview

,Metric,Value
0,Cleaned rows,392692
1,Cleaned columns,9
2,Unique customers,4338
3,Unique invoices,18532
4,Unique products,3665
5,Unique countries,37
6,Start date,2010-12-01 08:26:00
7,End date,2011-12-09 12:50:00
8,Total revenue,8887208.89


SAVE CLEANED DATASET

In [8]:
cleaned_df.to_csv(
    PROCESSED_DIR / "cleaned_online_retail.csv",
    index=False
)

print("Cleaned dataset saved successfully.")
print("File path:", PROCESSED_DIR / "cleaned_online_retail.csv")

Cleaned dataset saved successfully.
File path: ..\data\processed\cleaned_online_retail.csv


EXPLANATION

In [ ]:
print("DATA CLEANING SUMMARY")
print("---------------------")
print(f"Original records: {len(df):,}")
print(f"Cleaned records: {len(cleaned_df):,}")
print(f"Records removed: {len(df) - len(cleaned_df):,}")
print(f"Final customers available for segmentation: {cleaned_df['CustomerID'].nunique():,}")
print("The cleaned dataset is now ready for RFM and behavioural volatility feature engineering.")

DATA CLEANING SUMMARY
---------------------
Original records: 541,909
Cleaned records: 392,692
Records removed: 149,217
Final customers available for segmentation: 4,338
The cleaned dataset is now ready for RFM and behavioural volatility feature engineering.


: 